In [27]:
import os
from dotenv import load_dotenv
from langchain_huggingface import HuggingFaceEmbeddings

load_dotenv()
os.environ['HF_TOKEN']=os.getenv("HF_TOKEN")


embeddings=HuggingFaceEmbeddings(model_name="all-MiniLM-L6-v2")

c:\Users\Echo\Desktop\KNA_AgenticAI\venv\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [28]:
embed_result = embeddings.embed_query("Hello AI")

In [29]:
documents = ["What is the capital of USA?",
            "Who is the president of USA?",
            "Who is the prime minister of India?"
        ]

document_embedding = embeddings.embed_documents(documents)

In [30]:
my_query = "Narendra Modi is the prime minister of India."

query_embedding = embeddings.embed_query(my_query)

In [31]:
from sklearn.metrics.pairwise import cosine_similarity

cosine_similarity([query_embedding], document_embedding)

array([[0.07118422, 0.33584702, 0.76168938]])

In [32]:
from sklearn.metrics.pairwise import euclidean_distances

euclidean_distances([query_embedding], document_embedding)

array([[1.36294956, 1.15252155, 0.69037757]])

| Metric            | Similarity Score Range | Behavior                              |
| ----------------- | ---------------------- | ------------------------------------- |
| Cosine Similarity | \[-1, 1]               | Focuses on angle only |
| L2 Distance       | \[0, ∞)                | Focuses on **magnitude + direction**  |


In [33]:
import faiss
from langchain_community.vectorstores import FAISS
from langchain_community.docstore.in_memory import InMemoryDocstore

In [34]:
# from uuid import uuid4
from langchain_core.documents import Document

document_1 = Document(
    page_content="I had chocolate chip pancakes and scrambled eggs for breakfast this morning.",
    metadata={"source": "tweet"},
)

document_2 = Document(
    page_content="The weather forecast for tomorrow is cloudy and overcast, with a high of 62 degrees.",
    metadata={"source": "news"},
)

document_3 = Document(
    page_content="Building an exciting new project with LangChain - come check it out!",
    metadata={"source": "tweet"},
)

document_4 = Document(
    page_content="Robbers broke into the city bank and stole $1 million in cash.",
    metadata={"source": "news"},
)

document_5 = Document(
    page_content="Wow! That was an amazing movie. I can't wait to see it again.",
    metadata={"source": "tweet"},
)

document_6 = Document(
    page_content="Is the new iPhone worth the price? Read this review to find out.",
    metadata={"source": "website"},
)

document_7 = Document(
    page_content="The top 10 soccer players in the world right now.",
    metadata={"source": "website"},
)

document_8 = Document(
    page_content="LangGraph is the best framework for building stateful, agentic applications!",
    metadata={"source": "tweet"},
)

document_9 = Document(
    page_content="The stock market is down 500 points today due to fears of a recession.",
    metadata={"source": "news"},
)

document_10 = Document(
    page_content="I have a bad feeling I am going to get deleted :(",
    metadata={"source": "tweet"},
)

documents = [
    document_1,
    document_2,
    document_3,
    document_4,
    document_5,
    document_6,
    document_7,
    document_8,
    document_9,
    document_10,
]

In [35]:
index=faiss.IndexFlatIP(384)
vector_store=FAISS(
    embedding_function=embeddings,
    index=index,
    docstore=InMemoryDocstore(),
    index_to_docstore_id={},
)

In [36]:
vector_store.add_documents(documents=documents)

['e1f7c618-7be5-45a7-862e-eb4ab5d506af',
 'b4bd09d5-6fa5-4de1-836f-1889fbe82a70',
 '2c2d1bca-1d3b-4af3-b37f-3497fc760fec',
 'f84bb993-2846-429b-b964-f2d464cc2cc9',
 '53bbb7f4-a2a7-47ca-9656-052a04d7438e',
 '98e40bf4-70f8-418c-bafb-38ecbcb9ede2',
 'b94c938f-76cc-4176-9231-d2f47040fc3f',
 '3ebaa195-f842-42f3-8015-6c7ec91c471f',
 'f827e431-4877-4480-99be-b58f23ff2941',
 '601ae43a-2c71-4e78-8645-42876e1c05ef']

In [37]:
vector_store.similarity_search(
    "LangChain provides abstractions to make working with LLMs easy",
    k=2 #hyperparameter
    
)

[Document(id='2c2d1bca-1d3b-4af3-b37f-3497fc760fec', metadata={'source': 'tweet'}, page_content='Building an exciting new project with LangChain - come check it out!'),
 Document(id='3ebaa195-f842-42f3-8015-6c7ec91c471f', metadata={'source': 'tweet'}, page_content='LangGraph is the best framework for building stateful, agentic applications!')]

In [38]:
vector_store.similarity_search(
    "LangChain provides abstractions to make working with LLMs easy",
    #k=2 #hyperparameter,
    filter={"source":{"$eq": "tweet"}}
    
)

[Document(id='2c2d1bca-1d3b-4af3-b37f-3497fc760fec', metadata={'source': 'tweet'}, page_content='Building an exciting new project with LangChain - come check it out!'),
 Document(id='3ebaa195-f842-42f3-8015-6c7ec91c471f', metadata={'source': 'tweet'}, page_content='LangGraph is the best framework for building stateful, agentic applications!'),
 Document(id='601ae43a-2c71-4e78-8645-42876e1c05ef', metadata={'source': 'tweet'}, page_content='I have a bad feeling I am going to get deleted :('),
 Document(id='e1f7c618-7be5-45a7-862e-eb4ab5d506af', metadata={'source': 'tweet'}, page_content='I had chocolate chip pancakes and scrambled eggs for breakfast this morning.')]

In [40]:
result=vector_store.similarity_search(
    "LangChain provides abstractions to make working with LLMs easy",
    #k=2 #hyperparameter,
    filter={"source":"news"}
    
)
result

[Document(id='f84bb993-2846-429b-b964-f2d464cc2cc9', metadata={'source': 'news'}, page_content='Robbers broke into the city bank and stole $1 million in cash.'),
 Document(id='b4bd09d5-6fa5-4de1-836f-1889fbe82a70', metadata={'source': 'news'}, page_content='The weather forecast for tomorrow is cloudy and overcast, with a high of 62 degrees.'),
 Document(id='f827e431-4877-4480-99be-b58f23ff2941', metadata={'source': 'news'}, page_content='The stock market is down 500 points today due to fears of a recession.')]

In [41]:
retriever=vector_store.as_retriever(search_kwargs={"k": 3})

In [42]:
retriever.invoke("LangChain provides abstractions to make working with LLMs easy")

[Document(id='2c2d1bca-1d3b-4af3-b37f-3497fc760fec', metadata={'source': 'tweet'}, page_content='Building an exciting new project with LangChain - come check it out!'),
 Document(id='3ebaa195-f842-42f3-8015-6c7ec91c471f', metadata={'source': 'tweet'}, page_content='LangGraph is the best framework for building stateful, agentic applications!'),
 Document(id='601ae43a-2c71-4e78-8645-42876e1c05ef', metadata={'source': 'tweet'}, page_content='I have a bad feeling I am going to get deleted :(')]

In [43]:
vector_store.save_local("today's class faiss index")

In [44]:
new_vector_store=FAISS.load_local(
  "today's class faiss index",embeddings ,allow_dangerous_deserialization=True
)

In [45]:
new_vector_store.similarity_search("langchain")

[Document(id='2c2d1bca-1d3b-4af3-b37f-3497fc760fec', metadata={'source': 'tweet'}, page_content='Building an exciting new project with LangChain - come check it out!'),
 Document(id='3ebaa195-f842-42f3-8015-6c7ec91c471f', metadata={'source': 'tweet'}, page_content='LangGraph is the best framework for building stateful, agentic applications!'),
 Document(id='53bbb7f4-a2a7-47ca-9656-052a04d7438e', metadata={'source': 'tweet'}, page_content="Wow! That was an amazing movie. I can't wait to see it again."),
 Document(id='b94c938f-76cc-4176-9231-d2f47040fc3f', metadata={'source': 'website'}, page_content='The top 10 soccer players in the world right now.')]

In [ ]:
from langchain_community.document_loaders import PyPDFLoader
from langchain_text_splitters import RecursiveCharacterTextSplitter
from langchain_google_genai import ChatGoogleGenerativeAI
import pprint
from langchain import hub

FILE_PATH=r"llama2.pdf"
loader=PyPDFLoader(FILE_PATH)
pages=loader.load()

splitter = RecursiveCharacterTextSplitter(
    chunk_size=500,#hyperparameter
    chunk_overlap=50 #hyperparemeter
)
split_docs = splitter.split_documents(pages)

index=faiss.IndexFlatIP(384)
vector_store=FAISS(
    embedding_function=embeddings,
    index=index,
    docstore=InMemoryDocstore(),
    index_to_docstore_id={},
)

vector_store.add_documents(documents=split_docs)

retriever=vector_store.as_retriever(
    search_kwargs={"k": 10} #hyperparameter
)

retriever.invoke("what is llama model?")

model=ChatGoogleGenerativeAI(model='gemini-1.5-flash')

prompt = hub.pull("rlm/rag-prompt")
pprint.pprint(prompt.messages)


ValueError: File path llama2.pdf is not a valid file or url

In [49]:
from langchain_core.output_parsers import StrOutputParser
from langchain_core.runnables import RunnablePassthrough

def format_docs(docs):
    return "\n\n".join(doc.page_content for doc in docs)
    
rag_chain = (
    {"context": retriever | format_docs, "question": RunnablePassthrough()}
    | prompt
    | model
    | StrOutputParser()
)

rag_chain.invoke("what is llama model?")

NameError: name 'prompt' is not defined